In [1]:
!pip install transformers datasets evaluate sacrebleu bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [bitsandbytes]


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
import evaluate

# 1. Load Dataset (English to French)
# Using a small subset for computational feasibility
dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train[:5000]")
dataset = dataset.train_test_split(test_size=0.2)
train_data = dataset["train"]
test_data = dataset["test"]

# Define source and target languages
source_lang = "en"
target_lang = "fr"

def preprocess_function(examples, tokenizer, max_length=128):
    inputs = [ex[source_lang] for ex in examples["translation"]]
    targets = [ex[target_lang] for ex in examples["translation"]]

    # Modern Hugging Face API uses text_target directly
    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=max_length,
        truncation=True,
        padding="max_length"
    )

    return model_inputs


In [5]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

def train_encoder_decoder(model_checkpoint, train_data, test_data, is_mt5=False):
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

    # Prefix for mT5
    if is_mt5:
        def add_prefix(example):
            example['translation']['en'] = "translate English to French: " + example['translation']['en']
            return example
        train_data = train_data.map(add_prefix)
        test_data = test_data.map(add_prefix)

    tokenized_train = train_data.map(lambda x: preprocess_function(x, tokenizer), batched=True)
    tokenized_test = test_data.map(lambda x: preprocess_function(x, tokenizer), batched=True)

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./results_{model_checkpoint.split('/')[-1]}",
        eval_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=3,
        predict_with_generate=True,
        fp16=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time

    return model, tokenizer, training_time

# Execute Basic Transformer (using MarianMT as the basic encoder-decoder baseline)
print("Training Basic Transformer...")
transformer_model, transformer_tokenizer, trans_time = train_encoder_decoder("Helsinki-NLP/opus-mt-en-fr", train_data, test_data)

# Execute mT5
print("Fine-tuning mT5...")
mt5_model, mt5_tokenizer, mt5_time = train_encoder_decoder("google/mt5-small", train_data, test_data, is_mt5=True)

Training Basic Transformer...


/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import time
import torch
from transformers import AutoTokenizer

def generate_decoder_only_translations(model_id, test_dataset, num_samples=50):
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config, device_map="auto")

    translations = []
    inference_times = []

    # Slice the dataset to only process 'num_samples'
    subset = test_dataset.select(range(num_samples))

    print(f"\nStarting generation for {model_id}...")
    for i, example in enumerate(subset['translation']):
        if i % 10 == 0:
            print(f"Translating sentence {i}/{num_samples}")

        source_text = example['en']
        prompt = f"Translate the following English text to French. Only output the French translation.\nEnglish: {source_text}\nFrench:"

        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

        start_time = time.time()
        outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.3, do_sample=True)
        inf_time = time.time() - start_time  # <-- Fixed line

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translation = decoded.split("French:")[-1].strip()

        translations.append(translation)
        inference_times.append(inf_time)

    avg_inference_time = sum(inference_times) / len(inference_times)
    print(f"Finished {model_id}! Avg time per sentence: {avg_inference_time:.2f}s")
    return translations, avg_inference_time

# Run TinyLlama on just 50 samples
llama_translations, llama_inf_time = generate_decoder_only_translations("TinyLlama/TinyLlama-1.1B-Chat-v1.0", test_data, num_samples=50)

# Run Gemma (or Qwen if Gemma is gated) on 50 samples
gemma_translations, gemma_inf_time = generate_decoder_only_translations("Qwen/Qwen2.5-1.5B-Instruct", test_data, num_samples=50)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Starting generation for TinyLlama/TinyLlama-1.1B-Chat-v1.0...
Translating sentence 0/50


[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Translating sentence 10/50


...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Starting generation for Qwen/Qwen2.5-1.5B-Instruct...
Translating sentence 0/50
Translating sentence 10/50
Translating sentence 20/50
Translating sentence 30/50
Translating sentence 40/50
Finished Qwen/Qwen2.5-1.5B-Instruct! Avg time per sentence: 7.33s


In [6]:
import evaluate

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def evaluate_model(predictions, references):
    bleu_score = bleu.compute(predictions=predictions, references=[[r] for r in references])
    chrf_score = chrf.compute(predictions=predictions, references=[[r] for r in references])
    return bleu_score['score'], chrf_score['score']

# 1. Grab only the first 50 references to match our subset
subset_test_data = test_data.select(range(50))
references = [ex['fr'] for ex in subset_test_data['translation']]

# 2. Get predictions for encoder-decoder models on the same 50 sentences
def get_enc_dec_preds(model, tokenizer, test_dataset, is_mt5=False):
    preds = []
    model.to("cuda")
    for ex in test_dataset['translation']:
        text = ex['en']
        if is_mt5: text = "translate English to French: " + text
        inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True).to("cuda")
        outputs = model.generate(**inputs)
        preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))
    return preds

print("Generating Basic Transformer predictions...")
trans_preds = get_enc_dec_preds(transformer_model, transformer_tokenizer, subset_test_data)

print("Generating mT5 predictions...")
mt5_preds = get_enc_dec_preds(mt5_model, mt5_tokenizer, subset_test_data, is_mt5=True)

# 3. Calculate and print all scores
results = {
    "Basic Transformer": evaluate_model(trans_preds, references),
    "mT5": evaluate_model(mt5_preds, references),
    "LLaMA (TinyLlama)": evaluate_model(llama_translations, references),
    "Gemma (Qwen)": evaluate_model(gemma_translations, references)
}

print("\n--- FINAL EVALUATION SCORES ---")
print("Model | BLEU | chrF")
for model_name, scores in results.items():
    print(f"{model_name} | {scores[0]:.2f} | {scores[1]:.2f}")

Generating Basic Transformer predictions...
Generating mT5 predictions...


/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1963: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(



--- FINAL EVALUATION SCORES ---
Model | BLEU | chrF
Basic Transformer | 21.49 | 47.07
mT5 | 0.00 | 0.00
LLaMA (TinyLlama) | 2.52 | 28.47
Gemma (Qwen) | 2.46 | 26.20
